In [1]:
%pip install -U transformers datasets accelerate peft trl bitsandbytes

Defaulting to user installation because normal site-packages is not writeable
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.7/11.7 MB 134.3 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 559.1/559.1 kB 16.5 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 775.8/775.8 kB 23.3 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 925.8/925.8 kB 22.3 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 41.0/41.0 MB 101.0 MB/s  0:00:00m0:00:01
  Attempting uninstall: bitsandbytes
    Found existing installation: bitsandbytes 0.49.2
    Uninstalling bitsandbytes-0.49.2:
      Successfully uninstalled bitsandbytes-0.49.2
  Attempting uninstall: datasets━━━━━━━━━━━━━━━━ 0/5 [bitsandbytes]
    Found existing installation: datasets 5.0.0m 0/5 [bitsandbytes]
    Uninstalling datasets-5.0.0:━━━━━━━━━━━━ 0/5 [bitsandbytes]
      Successfully uninstalled datasets-5.0.0 0/5 [bitsandbytes]
   ━━━━━━━━╺━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1/5 [datasets]  WARNING: The script da

In [1]:
import torch
import transformers
import datasets
import peft
import trl
import bitsandbytes as bnb
from datasets import Dataset, ClassLabel
print("PyTorch:", torch.__version__)
print("Transformers:", transformers.__version__)
print("Datasets:", datasets.__version__)
print("PEFT:", peft.__version__)
print("TRL:", trl.__version__)
print("bitsandbytes:", bnb.__version__)
print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

[HAMI-core Msg(588:140076266478912:libvgpu.c:839)]: Initializing.....
[HAMI-core Msg(588:140076266478912:libvgpu.c:855)]: Initialized


PyTorch: 2.9.1+cu128
Transformers: 5.15.0
Datasets: 5.0.1
PEFT: 0.20.0
TRL: 1.10.0
bitsandbytes: 0.50.1
CUDA available: True
GPU: NVIDIA A100 80GB PCIe


In [2]:
from pathlib import Path

PROJECT_DIR = Path("/home/jovyan/project work/data_analyssis/fine tuning")

OUTPUT_DIR = PROJECT_DIR / "outputs"
HF_DATASET_PATH = OUTPUT_DIR / "prepared_dataset"
MODEL_OUTPUT_DIR = OUTPUT_DIR / "mistral_qlora"
RESULTS_DIR = OUTPUT_DIR / "results"
LOGS_DIR = OUTPUT_DIR / "logs"

TRAIN_JSONL_PATH = OUTPUT_DIR / "train_sft.jsonl"
EVAL_JSONL_PATH = OUTPUT_DIR / "eval_sft.jsonl"
CLEANED_DATA_PATH = OUTPUT_DIR / "cleaned_ruhsold_train.csv"

for directory in [
    OUTPUT_DIR,
    MODEL_OUTPUT_DIR,
    RESULTS_DIR,
    LOGS_DIR,
]:
    directory.mkdir(parents=True, exist_ok=True)

print("Prepared dataset path:", HF_DATASET_PATH)
print("Model output path:", MODEL_OUTPUT_DIR)

Prepared dataset path: /home/jovyan/project work/data_analyssis/fine tuning/outputs/prepared_dataset
Model output path: /home/jovyan/project work/data_analyssis/fine tuning/outputs/mistral_qlora


In [3]:
import pandas as pd

TRAIN_PATH = "../RUHSOLD_train.tsv"  # Change this path if necessary

train_df = pd.read_csv(
    TRAIN_PATH,
    sep="\t",
    header=None,
    names=["text", "label"]
)

train_df["text"] = train_df["text"].astype(str).str.strip()
train_df["label"] = train_df["label"].astype(int)

train_df = train_df.dropna(subset=["text", "label"])
train_df = train_df[train_df["text"].str.len() > 0]
train_df = train_df.drop_duplicates(subset=["text"]).reset_index(drop=True)

print("Training samples:", len(train_df))
print(train_df["label"].value_counts().sort_index())
train_df.head()

Training samples: 6401
label
0    1534
1    3422
2     500
3     535
4     410
Name: count, dtype: int64


,text,label
0,kia howa hai aap ko allah bless and protect yo...,1
1,randdi hai,3
2,smjh to agai thi mjhy,1
3,haan yrr tuny sahi thukayi ki abhi tak lund da...,3
4,rundi ka bacha bharwaaa ptm ka kuttaa,0


In [4]:
LABEL_CONFIG = {
    0: {
        "name": "Abusive/Offensive",
        "definition": (
            "A post containing insults, personal attacks, humiliation, "
            "or rude/offensive language directed at a person or group, "
        )
    },

    1: {
        "name": "Normal",
        "definition": (
            "A neutral or ordinary post that does not contain targeted abuse, "
            "untargeted profanity, sexism, or religious hate."
        )
    },

    2: {
        "name": "Religious Hate",
        "definition": (
            "A post expressing hatred, hostility, insult, stereotyping, "
            "or incitement against a person or group because of their "
            "religious beliefs or lack of religious beliefs."
        )
    },

    3: {
        "name": "Sexism",
        "definition": (
            "A post expressing hatred, hostility, insult, stereotyping, "
            "or degradation toward a person or group because of gender "
            "or sexual orientation."
        )
    },

    4: {
        "name": "Profane",
        "definition": (
            "A post containing vulgar, foul, obscene, or explicit profanity "
            "without an intended target."
        )
    }
}

In [5]:
INSTRUCTION_TEMPLATES = [
    (
        "You are generating synthetic examples for an academic Roman Urdu "
        "hate-speech and abusive-language classification dataset.\n\n"
        "Target class: {label}\n"
        "Class definition: {definition}\n\n"
        "Generate one natural, informal Roman Urdu social media post that "
        "clearly belongs to the target class. The post itself must express "
        "the requested category rather than discuss or condemn it.\n\n"
        "Write only in Roman Urdu using the Latin alphabet. Do not use Urdu "
        "or Arabic script. Return only the generated post without an "
        "explanation, label, quotation marks or additional text."
    )
]

In [6]:
def create_conversation(row, index):
    label_id = int(row["label"])
    label_info = LABEL_CONFIG[label_id]

    # Deterministic template selection for reproducibility
    template = INSTRUCTION_TEMPLATES[index % len(INSTRUCTION_TEMPLATES)]

    instruction = template.format(
        label=label_info["name"],
        definition=label_info["definition"]
    )

    return {
        "messages": [
            {
                "role": "user",
                "content": instruction
            },
            {
                "role": "assistant",
                "content": str(row["text"]).strip()
            }
        ],
        "label": label_id
    }


training_records = [
    create_conversation(row, index)
    for index, (_, row) in enumerate(train_df.iterrows())
]

sft_dataset = Dataset.from_list(training_records)

print(sft_dataset)
print("\nExample record:\n")
print(sft_dataset[0])

Dataset({
    features: ['messages', 'label'],
    num_rows: 6401
})

Example record:

{'messages': [{'role': 'user', 'content': 'You are generating synthetic examples for an academic Roman Urdu hate-speech and abusive-language classification dataset.\n\nTarget class: Normal\nClass definition: A neutral or ordinary post that does not contain targeted abuse, untargeted profanity, sexism, or religious hate.\n\nGenerate one natural, informal Roman Urdu social media post that clearly belongs to the target class. The post itself must express the requested category rather than discuss or condemn it.\n\nWrite only in Roman Urdu using the Latin alphabet. Do not use Urdu or Arabic script. Return only the generated post without an explanation, label, quotation marks or additional text.'}, {'role': 'assistant', 'content': 'kia howa hai aap ko allah bless and protect you  aameen'}], 'label': 1}


In [7]:
label_feature = ClassLabel(
    num_classes=5,
    names=[
        LABEL_CONFIG[0]["name"],
        LABEL_CONFIG[1]["name"],
        LABEL_CONFIG[2]["name"],
        LABEL_CONFIG[3]["name"],
        LABEL_CONFIG[4]["name"]
    ]
)

sft_dataset = sft_dataset.cast_column(
    "label",
    label_feature
)

print(sft_dataset.features)

Casting the dataset:   0%|          | 0/6401 [00:00<?, ? examples/s]

{'messages': List({'role': Value('string'), 'content': Value('string')}), 'label': ClassLabel(names=['Abusive/Offensive', 'Normal', 'Religious Hate', 'Sexism', 'Profane'])}


In [8]:
from datasets import Dataset
from collections import Counter


sft_split = sft_dataset.train_test_split(
    test_size=0.05,
    seed=42,
    stratify_by_column="label"
)

sft_train_dataset = sft_split["train"]
sft_eval_dataset = sft_split["test"]

print("SFT training samples:", len(sft_train_dataset))
print("SFT evaluation samples:", len(sft_eval_dataset))

print("\nTrain labels:")
print(Counter(sft_train_dataset["label"]))

print("\nEvaluation labels:")
print(Counter(sft_eval_dataset["label"]))

SFT training samples: 6080
SFT evaluation samples: 321

Train labels:
Counter({1: 3250, 0: 1457, 3: 508, 2: 475, 4: 390})

Evaluation labels:
Counter({1: 172, 0: 77, 3: 27, 2: 25, 4: 20})


In [9]:
import shutil

# Remove the previously saved processed dataset, if it exists
if HF_DATASET_PATH.exists():
    shutil.rmtree(HF_DATASET_PATH)

# Save the Hugging Face DatasetDict
sft_split.save_to_disk(str(HF_DATASET_PATH))

# Save train and evaluation splits as JSONL
sft_train_dataset.to_json(
    str(TRAIN_JSONL_PATH),
    orient="records",
    lines=True,
    force_ascii=False
)

sft_eval_dataset.to_json(
    str(EVAL_JSONL_PATH),
    orient="records",
    lines=True,
    force_ascii=False
)

# Save the cleaned original RUHSOLD dataframe
train_df.to_csv(
    CLEANED_DATA_PATH,
    index=False,
    encoding="utf-8"
)

print("Files saved successfully:\n")
print("Hugging Face dataset:", HF_DATASET_PATH)
print("Training JSONL:", TRAIN_JSONL_PATH)
print("Evaluation JSONL:", EVAL_JSONL_PATH)
print("Cleaned CSV:", CLEANED_DATA_PATH)

Saving the dataset (0/1 shards):   0%|          | 0/6080 [00:00<?, ? examples/s]

Saving the dataset (0/1 shards):   0%|          | 0/321 [00:00<?, ? examples/s]

Creating json from Arrow format:   0%|          | 0/7 [00:00<?, ?ba/s]

Creating json from Arrow format:   0%|          | 0/1 [00:00<?, ?ba/s]

Files saved successfully:

Hugging Face dataset: /home/jovyan/project work/data_analyssis/fine tuning/outputs/prepared_dataset
Training JSONL: /home/jovyan/project work/data_analyssis/fine tuning/outputs/train_sft.jsonl
Evaluation JSONL: /home/jovyan/project work/data_analyssis/fine tuning/outputs/eval_sft.jsonl
Cleaned CSV: /home/jovyan/project work/data_analyssis/fine tuning/outputs/cleaned_ruhsold_train.csv


In [10]:
for label_id in range(5):
    example_index = sft_dataset["label"].index(label_id)
    example = sft_dataset[example_index]

    print("=" * 100)
    print("LABEL:", label_id)
    print("CLASS:", LABEL_CONFIG[label_id]["name"])

    print("\nUSER INSTRUCTION:\n")
    print(example["messages"][0]["content"])

    print("\nASSISTANT RESPONSE:\n")
    print(example["messages"][1]["content"])
    print()

LABEL: 0
CLASS: Abusive/Offensive

USER INSTRUCTION:

You are generating synthetic examples for an academic Roman Urdu hate-speech and abusive-language classification dataset.

Target class: Abusive/Offensive
Class definition: A post containing insults, personal attacks, humiliation, or rude/offensive language directed at a person or group, 

Generate one natural, informal Roman Urdu social media post that clearly belongs to the target class. The post itself must express the requested category rather than discuss or condemn it.

Write only in Roman Urdu using the Latin alphabet. Do not use Urdu or Arabic script. Return only the generated post without an explanation, label, quotation marks or additional text.

ASSISTANT RESPONSE:

rundi ka bacha bharwaaa ptm ka kuttaa

LABEL: 1
CLASS: Normal

USER INSTRUCTION:

You are generating synthetic examples for an academic Roman Urdu hate-speech and abusive-language classification dataset.

Target class: Normal
Class definition: A neutral or ord

In [11]:
import json

experiment_config = {
    "base_model": "mistralai/Mistral-7B-Instruct-v0.2",
    "fine_tuning_method": "QLoRA supervised fine-tuning",
    "source_dataset": str(TRAIN_PATH),
    "number_of_cleaned_rows": len(train_df),
    "number_of_sft_train_rows": len(sft_train_dataset),
    "number_of_sft_eval_rows": len(sft_eval_dataset),
    "evaluation_fraction": 0.05,
    "split_seed": 42,
    "label_config": LABEL_CONFIG,
    "instruction_templates": INSTRUCTION_TEMPLATES
}

CONFIG_PATH = OUTPUT_DIR / "experiment_config.json"

with open(CONFIG_PATH, "w", encoding="utf-8") as file:
    json.dump(
        experiment_config,
        file,
        ensure_ascii=False,
        indent=4
    )

print("Configuration saved to:")
print(CONFIG_PATH)

Configuration saved to:
/home/jovyan/project work/data_analyssis/fine tuning/outputs/experiment_config.json


In [12]:
import torch

if torch.cuda.is_available():
    free_memory, total_memory = torch.cuda.mem_get_info()

    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"Free GPU memory:  {free_memory / 1024**3:.2f} GB")
    print(f"Total GPU memory: {total_memory / 1024**3:.2f} GB")
    print(f"Allocated by PyTorch: {torch.cuda.memory_allocated() / 1024**3:.2f} GB")
    print(f"Reserved by PyTorch:  {torch.cuda.memory_reserved() / 1024**3:.2f} GB")

GPU: NVIDIA A100 80GB PCIe
Free GPU memory:  15.19 GB
Total GPU memory: 16.00 GB
Allocated by PyTorch: 0.00 GB
Reserved by PyTorch:  0.00 GB


In [13]:
from pathlib import Path
import shutil

home = Path.home()
hf_cache = home / ".cache" / "huggingface"

disk_usage = shutil.disk_usage(home)

print(f"Disk total: {disk_usage.total / 1024**3:.2f} GB")
print(f"Disk used:  {disk_usage.used / 1024**3:.2f} GB")
print(f"Disk free:  {disk_usage.free / 1024**3:.2f} GB")
print(f"\nHugging Face cache: {hf_cache}")

Disk total: 24.52 GB
Disk used:  20.22 GB
Disk free:  4.28 GB

Hugging Face cache: /home/jovyan/.cache/huggingface


In [14]:
from pathlib import Path

hub_cache = Path.home() / ".cache" / "huggingface" / "hub"

def directory_size_gb(path):
    total_bytes = sum(
        file.stat().st_size
        for file in path.rglob("*")
        if file.is_file()
    )
    return total_bytes / 1024**3

for model_path in sorted(hub_cache.glob("models--*")):
    print(
        f"{model_path.name}: "
        f"{directory_size_gb(model_path):.2f} GB"
    )

models--FacebookAI--xlm-roberta-base: 2.10 GB
models--intfloat--multilingual-e5-large: 4.21 GB
models--mistralai--Mistral-7B-Instruct-v0.2: 26.98 GB


In [15]:
import torch

MODEL_ID = "mistralai/Mistral-7B-Instruct-v0.2"
MAX_LENGTH = 320

LORA_R = 16
LORA_ALPHA = 32
LORA_DROPOUT = 0.05

print("Model:", MODEL_ID)
print("Maximum sequence length:", MAX_LENGTH)
print("Compute dtype:", torch.bfloat16)

Model: mistralai/Mistral-7B-Instruct-v0.2
Maximum sequence length: 320
Compute dtype: torch.bfloat16


In [16]:
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained(
    MODEL_ID,
    use_fast=True
)

if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

tokenizer.padding_side = "right"

print("Tokenizer loaded successfully.")
print("Padding token:", tokenizer.pad_token)
print("EOS token:", tokenizer.eos_token)

Tokenizer loaded successfully.
Padding token: </s>
EOS token: </s>


In [18]:
from transformers import BitsAndBytesConfig

quantization_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_use_double_quant=True
)

print(quantization_config)

BitsAndBytesConfig {
  "_load_in_4bit": true,
  "_load_in_8bit": false,
  "bnb_4bit_compute_dtype": "bfloat16",
  "bnb_4bit_quant_storage": "uint8",
  "bnb_4bit_quant_type": "nf4",
  "bnb_4bit_use_double_quant": true,
  "llm_int8_enable_fp32_cpu_offload": false,
  "llm_int8_has_fp16_weight": false,
  "llm_int8_skip_modules": null,
  "llm_int8_threshold": 6.0,
  "load_in_4bit": true,
  "load_in_8bit": false,
  "quant_method": "bitsandbytes"
}



In [19]:
# ============================================================
# HAMI WORKAROUND:
# DISABLE TRANSFORMERS CUDA ALLOCATOR WARMUP
# ============================================================

import gc
import torch
import transformers.modeling_utils as modeling_utils

gc.collect()

if torch.cuda.is_available():
    torch.cuda.empty_cache()


# ------------------------------------------------------------
# Save original function in case we want to restore it later
# ------------------------------------------------------------

_original_caching_allocator_warmup = (
    modeling_utils.caching_allocator_warmup
)


# ------------------------------------------------------------
# Replace allocator warmup with a no-op
# ------------------------------------------------------------

def _skip_caching_allocator_warmup(
    *args,
    **kwargs
):
    return None


modeling_utils.caching_allocator_warmup = (
    _skip_caching_allocator_warmup
)


print(
    "Transformers caching_allocator_warmup disabled."
)

free, total = torch.cuda.mem_get_info()

print(
    f"CUDA free : {free / 1024**3:.2f} GB"
)

print(
    f"CUDA total: {total / 1024**3:.2f} GB"
)

Transformers caching_allocator_warmup disabled.
CUDA free : 15.19 GB
CUDA total: 16.00 GB


In [20]:
# ============================================================
# LOAD 4-BIT MISTRAL
# ============================================================

from transformers import (
    AutoModelForCausalLM,
    BitsAndBytesConfig,
)


quantization_config = BitsAndBytesConfig(

    load_in_4bit=True,

    bnb_4bit_quant_type="nf4",

    bnb_4bit_compute_dtype=torch.bfloat16,

    bnb_4bit_use_double_quant=True,
)


base_model = (
    AutoModelForCausalLM
    .from_pretrained(

        MODEL_ID,

        quantization_config=quantization_config,

        device_map="auto",

        dtype=torch.bfloat16,

        low_cpu_mem_usage=True,
    )
)


base_model.config.use_cache = False


print(
    "Mistral loaded successfully."
)

print(
    "Device map:",
    base_model.hf_device_map
)

print(
    "Memory footprint:",
    f"{base_model.get_memory_footprint() / 1024**3:.2f} GB"
)

Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

[HAMI-core ERROR (pid:588 thread=140076266478912 allocator.c:121)]: cuMemoryAllocate failed res=2
[HAMI-core ERROR (pid:588 thread=140076266478912 allocator.c:121)]: cuMemoryAllocate failed res=2
[HAMI-core ERROR (pid:588 thread=140076266478912 allocator.c:121)]: cuMemoryAllocate failed res=2
[HAMI-core ERROR (pid:588 thread=140076266478912 allocator.c:121)]: cuMemoryAllocate failed res=2


OutOfMemoryError: CUDA out of memory. Tried to allocate 112.00 MiB. GPU 0 has a total capacity of 16.00 GiB of which 13.62 GiB is free. Process 36015 has 3.54 GiB memory in use. Process 69754 has 68.69 GiB memory in use. Process 81193 has 3.75 GiB memory in use. Process 161289 has 600.00 MiB memory in use. Process 162265 has 416.00 MiB memory in use. Process 162880 has 2.04 GiB memory in use. Of the allocated memory 1.54 GiB is allocated by PyTorch, and 18.52 MiB is reserved by PyTorch but unallocated. If reserved but unallocated memory is large try setting PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True to avoid fragmentation.  See documentation for Memory Management  (https://pytorch.org/docs/stable/notes/cuda.html#environment-variables)

In [23]:
base_model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    quantization_config=quantization_config,
    device_map={"": 0},
    dtype=torch.bfloat16,
    low_cpu_mem_usage=True
)

[HAMI-core ERROR (pid:133 thread=140691244584256 allocator.c:121)]: cuMemoryAllocate failed res=2
[HAMI-core ERROR (pid:133 thread=140691244584256 allocator.c:121)]: cuMemoryAllocate failed res=2


OutOfMemoryError: CUDA out of memory. Tried to allocate 3.74 GiB. GPU 0 has a total capacity of 16.00 GiB of which 15.59 GiB is free. Process 36015 has 3.54 GiB memory in use. Process 69754 has 68.69 GiB memory in use. Process 81193 has 3.75 GiB memory in use. Process 160047 has 416.00 MiB memory in use. Of the allocated memory 0 bytes is allocated by PyTorch, and 0 bytes is reserved by PyTorch but unallocated. If reserved but unallocated memory is large try setting PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True to avoid fragmentation.  See documentation for Memory Management  (https://pytorch.org/docs/stable/notes/cuda.html#environment-variables)

In [20]:
def show_gpu_memory(stage):
    free_memory, total_memory = torch.cuda.mem_get_info()

    print(f"\nGPU memory after {stage}:")
    print(f"Free:      {free_memory / 1024**3:.2f} GB")
    print(f"Used:      {(total_memory - free_memory) / 1024**3:.2f} GB")
    print(f"Allocated: {torch.cuda.memory_allocated() / 1024**3:.2f} GB")
    print(f"Reserved:  {torch.cuda.memory_reserved() / 1024**3:.2f} GB")


show_gpu_memory("loading the 4-bit base model")


GPU memory after loading the 4-bit base model:
Free:      7.36 GB
Used:      8.64 GB
Allocated: 7.20 GB
Reserved:  7.83 GB


In [21]:
from peft import prepare_model_for_kbit_training

base_model = prepare_model_for_kbit_training(
    base_model,
    use_gradient_checkpointing=True
)

base_model.gradient_checkpointing_enable()
base_model.config.use_cache = False

print("Model prepared for k-bit training.")
print("Gradient checkpointing:", base_model.is_gradient_checkpointing)

Model prepared for k-bit training.
Gradient checkpointing: True


In [22]:
from peft import LoraConfig, TaskType, get_peft_model

LORA_R = 16
LORA_ALPHA = 32
LORA_DROPOUT = 0.05

target_modules = [
    "q_proj",
    "k_proj",
    "v_proj",
    "o_proj",
    "gate_proj",
    "up_proj",
    "down_proj",
]

lora_config = LoraConfig(
    r=LORA_R,
    lora_alpha=LORA_ALPHA,
    lora_dropout=LORA_DROPOUT,
    bias="none",
    task_type=TaskType.CAUSAL_LM,
    target_modules=target_modules,
)

model = get_peft_model(
    base_model,
    lora_config
)

model.print_trainable_parameters()

trainable params: 41,943,040 || all params: 7,283,675,136 || trainable%: 0.5758


In [23]:
show_gpu_memory("preparing the model and attaching LoRA")


GPU memory after preparing the model and attaching LoRA:
Free:      6.72 GB
Used:      9.28 GB
Allocated: 7.85 GB
Reserved:  8.47 GB


In [24]:
import inspect
from trl import SFTConfig, SFTTrainer

print(inspect.signature(SFTConfig))

(output_dir: str | None = None, per_device_train_batch_size: int = 8, num_train_epochs: float = 3.0, max_steps: int = -1, learning_rate: float = 2e-05, lr_scheduler_type: transformers.trainer_utils.SchedulerType | str = 'linear', lr_scheduler_kwargs: dict | str | None = None, warmup_steps: float = 0, optim: transformers.training_args.OptimizerNames | str = 'adamw_torch_fused', optim_args: str | None = None, weight_decay: float = 0.0, adam_beta1: float = 0.9, adam_beta2: float = 0.999, adam_epsilon: float = 1e-08, optim_target_modules: None | str | list[str] = None, gradient_accumulation_steps: int = 1, average_tokens_across_devices: bool = True, max_grad_norm: float = 1.0, label_smoothing_factor: float = 0.0, bf16: bool | None = None, fp16: bool = False, bf16_full_eval: bool = False, fp16_full_eval: bool = False, tf32: bool | None = None, gradient_checkpointing: bool = True, gradient_checkpointing_kwargs: dict[str, typing.Any] | str | None = None, torch_compile: bool = False, torch_com

In [25]:
sample_messages = sft_train_dataset[0]["messages"]

try:
    encoded_chat = tokenizer.apply_chat_template(
        sample_messages,
        tokenize=True,
        return_dict=True,
        return_assistant_tokens_mask=True,
        add_generation_prompt=False
    )

    print("Returned keys:", encoded_chat.keys())

    assistant_mask = encoded_chat.get("assistant_masks")

    if assistant_mask is None:
        assistant_mask = encoded_chat.get("assistant_mask")

    if assistant_mask is not None:
        print("Assistant mask supported.")
        print("Total tokens:", len(encoded_chat["input_ids"]))
        print("Assistant tokens:", sum(assistant_mask))
    else:
        print("No assistant mask returned.")

except Exception as error:
    print("Assistant mask test failed:")
    print(type(error).__name__, error)

[transformers] return_assistant_tokens_mask==True but chat template does not contain `{% generation %}` keyword.


Returned keys: KeysView({'input_ids': [1, 733, 16289, 28793, 995, 460, 20365, 26735, 9254, 354, 396, 11860, 8325, 11149, 670, 7665, 28733, 14500, 5295, 304, 534, 6657, 28733, 11904, 16776, 13466, 28723, 13, 13, 5332, 875, 28747, 2484, 6657, 442, 17381, 3842, 13, 2472, 7526, 28747, 330, 1704, 8707, 17441, 28713, 28725, 3327, 10813, 28725, 1997, 22758, 28725, 442, 17381, 3842, 28725, 1671, 13395, 2718, 288, 10048, 442, 11487, 28723, 13, 13, 23342, 624, 4229, 28725, 5227, 282, 8325, 11149, 670, 2809, 28733, 9660, 1704, 369, 6315, 17827, 298, 272, 2718, 875, 28723, 415, 1704, 3837, 1580, 4072, 272, 11939, 8011, 3210, 821, 3342, 442, 19048, 28711, 378, 28723, 13, 13, 5238, 865, 297, 8325, 11149, 670, 1413, 272, 13729, 389, 26311, 28723, 2378, 459, 938, 11149, 670, 442, 9111, 294, 6767, 28723, 4571, 865, 272, 7138, 1704, 1671, 396, 13268, 28725, 3870, 28725, 17528, 352, 14191, 442, 4870, 2245, 28723, 733, 28748, 16289, 28793, 287, 540, 22219, 287, 849, 5044, 24393, 339, 446, 24393, 339, 2], 

In [26]:
training_args = SFTConfig(
    output_dir=str(MODEL_OUTPUT_DIR),

    max_steps=10,

    per_device_train_batch_size=1,
    gradient_accumulation_steps=8,

    learning_rate=2e-4,
    warmup_steps=1,
    weight_decay=0.01,

    bf16=True,
    fp16=False,

    gradient_checkpointing=True,
    gradient_checkpointing_kwargs={
        "use_reentrant": False
    },

    optim="paged_adamw_8bit",

    max_length=MAX_LENGTH,
    packing=False,

    logging_steps=1,

    eval_strategy="no",
    save_strategy="no",

    report_to="none",
    seed=42,
    remove_unused_columns=False,
    assistant_only_loss=False
)

print("Configured max steps:", training_args.max_steps)

Configured max steps: 10


[HAMI-core Msg(836:140105487828800:multiprocess_memory_limit.c:455)]: Calling exit handler 836


In [27]:
trainer = SFTTrainer(
    model=model,
    args=training_args,
    train_dataset=sft_train_dataset,
    processing_class=tokenizer
)

print("Trainer max steps:", trainer.args.max_steps)

Tokenizing train dataset:   0%|          | 0/6080 [00:00<?, ? examples/s]

Building labels for train dataset:   0%|          | 0/6080 [00:00<?, ? examples/s]

Truncating train dataset:   0%|          | 0/6080 [00:00<?, ? examples/s]

Dropping fully masked examples from train dataset:   0%|          | 0/6080 [00:00<?, ? examples/s]

Trainer max steps: 10


In [28]:
print("training_args max_steps:", training_args.max_steps)
print("trainer max_steps:", trainer.args.max_steps)

training_args max_steps: 10
trainer max_steps: 10


In [29]:
torch.cuda.reset_peak_memory_stats()

smoke_result = trainer.train()

print(smoke_result)
print(
    "Peak allocated GPU memory:",
    f"{torch.cuda.max_memory_allocated() / 1024**3:.2f} GB"
)

show_gpu_memory("10-step smoke test")

[transformers] The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'pad_token_id': 2}.


Step,Training Loss
1,4.431530
2,4.330018
3,2.907792
4,2.264786
5,1.753665
6,1.279473
7,0.827037
8,0.842605
9,0.964421
10,0.595629


TrainOutput(global_step=10, training_loss=2.0196956217288973, metrics={'train_runtime': 52.8057, 'train_samples_per_second': 1.515, 'train_steps_per_second': 0.189, 'total_flos': 633949521149952.0, 'train_loss': 2.0196956217288973, 'epoch': 0.013157894736842105})
Peak allocated GPU memory: 5.04 GB

GPU memory after 10-step smoke test:
Free:      6.82 GB
Used:      9.18 GB
Allocated: 4.47 GB
Reserved:  8.31 GB


In [30]:
from trl import SFTConfig

training_args = SFTConfig(
    output_dir=str(MODEL_OUTPUT_DIR),

    # Full training run
    num_train_epochs=1,
    max_steps=-1,

    per_device_train_batch_size=1,
    per_device_eval_batch_size=1,
    gradient_accumulation_steps=8,

    learning_rate=2e-4,
    warmup_steps=23,
    weight_decay=0.01,

    bf16=True,
    fp16=False,

    gradient_checkpointing=True,
    gradient_checkpointing_kwargs={
        "use_reentrant": False
    },

    optim="paged_adamw_8bit",

    max_length=MAX_LENGTH,
    packing=False,

    logging_steps=10,
    logging_first_step=True,

    eval_strategy="steps",
    eval_steps=100,

    save_strategy="steps",
    save_steps=100,
    save_total_limit=1,

    # Saves adapter/model only, not large optimizer states
    save_only_model=True,

    report_to="none",
    seed=42,
    data_seed=42,

    remove_unused_columns=False,

    # Mistral template returned an all-zero assistant mask
    assistant_only_loss=False
)

print("Epochs:", training_args.num_train_epochs)
print("Maximum steps:", training_args.max_steps)
print("Evaluation every:", training_args.eval_steps, "steps")
print("Saving every:", training_args.save_steps, "steps")

Epochs: 1
Maximum steps: -1
Evaluation every: 100 steps
Saving every: 100 steps


[HAMI-core Msg(841:140136315524928:multiprocess_memory_limit.c:455)]: Calling exit handler 841


In [35]:
print(train_df["label"].value_counts().sort_index())

label
0    1534
1    3422
2     500
3     535
4     410
Name: count, dtype: int64


In [34]:
# ============================================================
# SFT training class distribution
# ============================================================

train_distribution = (
    pd.Series(sft_train_dataset["label"])
    .value_counts()
    .sort_index()
)

eval_distribution = (
    pd.Series(sft_eval_dataset["label"])
    .value_counts()
    .sort_index()
)

distribution_table = pd.DataFrame({
    "label_id": train_distribution.index,
    "train_count": train_distribution.values,
    "eval_count": eval_distribution.values
})

distribution_table["class_name"] = distribution_table["label_id"].map(
    {
        0: "Abusive/Offensive",
        1: "Normal",
        2: "Religious Hate",
        3: "Sexism",
        4: "Profane"
    }
)

distribution_table

,label_id,train_count,eval_count,class_name
0,0,1457,77,Abusive/Offensive
1,1,3250,172,Normal
2,2,475,25,Religious Hate
3,3,508,27,Sexism
4,4,390,20,Profane


In [31]:
from trl import SFTTrainer

trainer = SFTTrainer(
    model=model,
    args=training_args,
    train_dataset=sft_train_dataset,
    eval_dataset=sft_eval_dataset,
    processing_class=tokenizer
)

print("Trainer created.")
print("Epochs:", trainer.args.num_train_epochs)
print("Max steps:", trainer.args.max_steps)
print("Evaluation strategy:", trainer.args.eval_strategy)
print("Save-only-model:", trainer.args.save_only_model)

Tokenizing train dataset:   0%|          | 0/6080 [00:00<?, ? examples/s]

Building labels for train dataset:   0%|          | 0/6080 [00:00<?, ? examples/s]

Truncating train dataset:   0%|          | 0/6080 [00:00<?, ? examples/s]

Dropping fully masked examples from train dataset:   0%|          | 0/6080 [00:00<?, ? examples/s]

Tokenizing eval dataset:   0%|          | 0/321 [00:00<?, ? examples/s]

Building labels for eval dataset:   0%|          | 0/321 [00:00<?, ? examples/s]

Truncating eval dataset:   0%|          | 0/321 [00:00<?, ? examples/s]

Dropping fully masked examples from eval dataset:   0%|          | 0/321 [00:00<?, ? examples/s]

Trainer created.
Epochs: 1
Max steps: -1
Evaluation strategy: IntervalStrategy.STEPS
Save-only-model: True


In [32]:
model.print_trainable_parameters()
show_gpu_memory("before full training")

trainable params: 41,943,040 || all params: 7,283,675,136 || trainable%: 0.5758

GPU memory after before full training:
Free:      6.82 GB
Used:      9.18 GB
Allocated: 4.43 GB
Reserved:  8.31 GB


In [33]:
import torch
import time

torch.cuda.reset_peak_memory_stats()

start_time = time.time()

train_result = trainer.train()

training_minutes = (time.time() - start_time) / 60

print(train_result)
print(f"\nTraining duration: {training_minutes:.2f} minutes")
print(
    "Peak allocated GPU memory:",
    f"{torch.cuda.max_memory_allocated() / 1024**3:.2f} GB"
)

show_gpu_memory("one-epoch training")

Step,Training Loss,Validation Loss


KeyboardInterrupt: 

In [35]:
FINAL_ADAPTER_DIR = MODEL_OUTPUT_DIR / "final_adapter"

trainer.save_model(str(FINAL_ADAPTER_DIR))
tokenizer.save_pretrained(str(FINAL_ADAPTER_DIR))

print("Final LoRA adapter saved to:")
print(FINAL_ADAPTER_DIR)

Final LoRA adapter saved to:
/home/jovyan/project work/data_analyssis/fine tuning/outputs/mistral_qlora/final_adapter


In [36]:
import json

metrics = train_result.metrics
metrics["training_duration_minutes"] = training_minutes
metrics["peak_gpu_memory_gb"] = (
    torch.cuda.max_memory_allocated() / 1024**3
)

METRICS_PATH = RESULTS_DIR / "one_epoch_training_metrics.json"

with open(METRICS_PATH, "w", encoding="utf-8") as file:
    json.dump(metrics, file, indent=4)

print("Training metrics saved to:")
print(METRICS_PATH)

Training metrics saved to:
/home/jovyan/project work/data_analyssis/fine tuning/outputs/results/one_epoch_training_metrics.json


In [37]:
print(sft_train_dataset[0])

{'messages': [{'role': 'user', 'content': 'You are generating synthetic examples for an academic Roman Urdu hate-speech and abusive-language classification dataset.\n\nTarget class: Abusive or offensive language\nClass definition: A post containing insults, personal attacks, humiliation, or offensive language, without primarily targeting religion or gender.\n\nGenerate one natural, informal Roman Urdu social-media post that clearly belongs to the target class. The post itself must express the requested category rather than discuss or condemn it.\n\nWrite only in Roman Urdu using the Latin alphabet. Do not use Urdu or Arabic script. Return only the generated post without an explanation, label, quotation marks or additional text.'}, {'role': 'assistant', 'content': 'bhenchod bikao saray k saray'}], 'label': 0}
